# Data merge and feature engineering


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [2]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [3]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [4]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
226584,2023-03-24 10:00:00,2,49.2336,28.4486,14.1,4.4,53.9,0.000,0.0,0.00,...,10,Vinnytsia,Europe/Kiev,2023-03-24,06:01:28,18:24:39,10:00:00,Вінницька,0,0.000000
514611,2024-08-05 13:00:00,5,48.0020,37.8145,20.1,16.5,80.3,9.800,100.0,41.67,...,13,Donetsk,Europe/Kiev,2024-08-05,05:10:31,19:58:09,13:00:00,Донецька,0,0.000000
478674,2024-06-04 03:00:00,21,46.6401,32.6142,23.8,15.0,62.2,0.300,100.0,8.33,...,3,Kherson,Europe/Kiev,2024-06-04,04:58:12,20:38:14,03:00:00,Херсонська,0,0.000000
464004,2024-05-09 16:00:00,15,46.4725,30.7371,14.2,4.1,52.9,0.000,0.0,0.00,...,16,Odesa,Europe/Kiev,2024-05-09,05:30:47,20:17:06,16:00:00,Одеська,1,25.033333
123643,2022-09-26 16:00:00,22,49.4168,26.9743,12.7,9.7,82.1,3.100,100.0,8.33,...,16,Khmelnytskyi,Europe/Kiev,2022-09-26,07:04:05,19:02:03,16:00:00,Хмельницька,0,0.000000
395688,2024-01-12 01:00:00,2,49.2336,28.4486,-8.6,-12.6,73.4,0.300,100.0,4.17,...,1,Vinnytsia,Europe/Kiev,2024-01-12,07:58:15,16:30:45,01:00:00,Вінницька,0,0.000000
299480,2023-07-29 00:00:00,10,50.4506,30.5243,20.8,14.4,67.5,0.400,100.0,12.50,...,0,Kyiv,Europe/Kiev,2023-07-29,05:20:46,20:47:14,00:00:00,Київська,0,0.000000
236341,2023-04-10 09:00:00,16,49.5879,34.5517,11.3,3.2,61.5,0.100,100.0,4.17,...,9,Poltava,Europe/Kiev,2023-04-10,06:00:50,19:26:37,09:00:00,Полтавська,0,0.000000
395425,2024-01-11 14:00:00,3,50.7469,25.3263,-4.4,-6.7,83.8,1.000,100.0,4.17,...,14,Lutsk,Europe/Kiev,2024-01-11,08:17:41,16:35:32,14:00:00,Волинська,1,46.800000
448891,2024-04-13 10:00:00,22,49.4168,26.9743,14.0,8.1,69.1,0.400,100.0,16.67,...,10,Khmelnytskyi,Europe/Kiev,2024-04-13,06:23:40,20:02:28,10:00:00,Хмельницька,0,0.000000


In [5]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,1,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,01:00:00,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,2,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,02:00:00,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,3,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,03:00:00,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,4,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,04:00:00,Вінницька,0,0.0


In [6]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634680 entries, 0 to 634679
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  634680 non-null  datetime64[us]
 1   region_id                      634680 non-null  int64         
 2   city_latitude                  634680 non-null  float64       
 3   city_longitude                 634680 non-null  float64       
 4   day_temp                       634680 non-null  float64       
 5   day_dew                        634680 non-null  float64       
 6   day_humidity                   634680 non-null  float64       
 7   day_precip                     634680 non-null  float64       
 8   day_precipprob                 634680 non-null  float64       
 9   day_precipcover                634680 non-null  float64       
 10  day_snow                       634680 non-null  float64       
 11  day_snowdepth   

In [7]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
185597,2023-01-12 06:00:00,7,48.62636,22.28514,3.0,2.0,93.1,0.000,0.0,0.00,...,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,0.0
203166,2023-02-11 18:00:00,8,47.82890,35.16260,-1.7,-5.3,77.1,0.000,0.0,0.00,...,1,17.266667,0.0,0.0,0.0,0.0,7.0,1,0,0.0
431570,2024-03-14 08:00:00,4,48.47530,35.01600,3.4,0.3,80.8,0.500,100.0,8.33,...,1,41.166667,1.0,1.0,1.0,0.0,12.0,0,0,5.0
182810,2023-01-07 10:00:00,4,48.47530,35.01600,-9.8,-15.3,64.7,0.055,100.0,8.33,...,0,0.000000,0.0,0.0,0.0,0.0,5.0,1,0,0.0
83780,2022-07-19 11:00:00,23,49.44070,32.06370,19.2,11.7,63.7,0.000,0.0,0.00,...,1,29.266667,0.0,0.0,0.0,1.0,4.0,0,0,0.0
485175,2024-06-15 10:00:00,18,50.90800,34.79720,19.3,15.9,81.2,12.677,100.0,8.33,...,0,0.000000,1.0,0.0,1.0,0.0,11.0,1,0,24.0
121908,2022-09-23 16:00:00,15,46.47250,30.73710,13.0,7.5,71.1,0.000,0.0,0.00,...,0,0.000000,1.0,0.0,0.0,0.0,4.0,0,0,5.0
169185,2022-12-14 18:00:00,11,48.50850,32.26560,-4.2,-6.3,85.1,0.000,0.0,0.00,...,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,0.0
401581,2024-01-22 06:00:00,16,49.58790,34.55170,-4.8,-7.7,80.8,0.000,0.0,0.00,...,0,0.000000,0.0,0.0,0.0,0.0,4.0,0,1,0.0
120510,2022-09-21 06:00:00,8,47.82890,35.16260,12.7,6.8,69.5,0.700,100.0,4.17,...,0,0.000000,0.0,0.0,0.0,0.0,4.0,0,1,3.0


In [8]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [9]:
def calculate_hours_since_last(series):
    series = series.shift(1).fillna(0)
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last))

In [10]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
239601,2022-05-01 13:00:00,11,48.50850,32.26560,11.3,4.9,66.4,1.000,100.0,33.33,...,0.0,0.0,0.0,0.0,3.0,1,0,0.0,0.0,13
251437,2023-09-06 18:00:00,11,48.50850,32.26560,19.5,10.6,60.5,0.000,0.0,0.00,...,0.0,0.0,0.0,1.0,6.0,0,0,5.0,1.0,7
198820,2023-09-18 03:00:00,9,48.92260,24.71470,19.4,14.1,74.4,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,2.0,0,1,3.0,0.0,20
560827,2022-10-10 11:00:00,24,48.29240,25.93550,9.5,4.7,74.4,0.117,100.0,4.17,...,1.0,1.0,0.0,0.0,6.0,0,0,24.0,4.0,0
537258,2023-02-07 07:00:00,23,49.44070,32.06370,-3.9,-8.1,73.1,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0,0,7.0,2.0,45
126274,2024-06-27 01:00:00,6,50.25360,28.66540,22.6,15.0,66.0,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0,1,6.0,1.0,30
137309,2022-09-23 21:00:00,7,48.62636,22.28514,9.5,6.1,80.4,0.595,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,0,6.0,0.0,32
209135,2024-11-20 23:00:00,9,48.92260,24.71470,7.2,2.7,74.4,7.600,100.0,41.67,...,0.0,0.0,0.0,0.0,0.0,0,1,5.0,0.0,53
305297,2023-10-17 04:00:00,14,46.97340,31.98520,7.5,2.9,74.5,0.000,0.0,0.00,...,1.0,1.0,0.0,0.0,5.0,0,1,5.0,4.0,0
250639,2023-08-04 12:00:00,11,48.50850,32.26560,24.9,16.5,63.1,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,3.0,0,0,2.0,0.0,13


In [11]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
348,2023-02-09,0.764517,-0.207114,-0.139333,-0.009408,0.050488,0.040747,-0.027254,0.090199,-0.062695,...,0.011915,0.004864,0.042241,-0.001829,-0.002709,-0.015370,0.010852,-0.025240,-0.015311,-0.027619
625,2023-11-13,0.833604,-0.135799,-0.096263,-0.156144,0.016985,-0.119410,0.067336,-0.085123,0.113967,...,0.020216,0.007193,-0.005431,0.000271,0.015544,-0.011767,0.013709,0.018157,-0.011122,-0.043632
1091,2025-02-28,0.786310,0.229240,-0.059217,0.099700,0.009861,-0.004482,-0.014747,0.066648,0.007473,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
909,2024-08-26,0.771310,0.051831,-0.035369,0.004643,-0.185376,0.077734,-0.011885,0.105130,0.236144,...,-0.001035,-0.033217,-0.009082,-0.001757,-0.015554,0.009644,0.011421,-0.010696,-0.020813,-0.009568
346,2023-02-07,0.779025,-0.216628,-0.087908,-0.001955,0.086726,0.050858,-0.028388,0.079886,-0.063636,...,-0.004269,-0.008542,-0.051097,0.004540,-0.032763,-0.027701,0.033449,0.044274,0.030095,-0.013358
888,2024-08-05,0.774232,0.022455,-0.119043,0.018077,-0.177582,0.082923,-0.000018,0.089700,0.222799,...,0.029711,-0.012546,0.016533,0.003538,0.003193,0.010511,-0.003366,-0.012609,0.006198,-0.005035
83,2022-05-17,0.686528,-0.204537,0.333230,0.077116,0.040664,-0.010572,0.027843,-0.089657,0.003849,...,0.000457,0.010644,-0.028188,0.020445,0.000297,0.031009,0.012033,0.015465,0.011992,-0.013441
1131,2025-04-09,0.808207,0.155231,0.050069,0.167195,-0.055109,-0.078699,-0.036779,-0.079267,-0.049175,...,0.020483,0.006579,-0.003340,-0.007572,0.043121,0.015244,-0.027504,0.002972,0.010520,0.017914
734,2024-03-04,0.750992,-0.087737,-0.220501,0.216754,0.008822,-0.085672,-0.043717,0.104343,0.088946,...,-0.004128,-0.012641,0.028193,0.002355,-0.016959,-0.001301,-0.008136,0.006608,-0.029713,0.010684
1414,2026-01-20,0.729710,0.286720,0.051573,-0.034337,0.289196,0.213785,-0.124872,-0.051395,0.002813,...,0.006608,-0.019808,0.009680,0.003025,0.003934,-0.004011,0.002936,0.002568,0.009158,0.028682


In [12]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [13]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [14]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [15]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
53483,2022-03-17 17:00:00,4,48.4753,35.0160,-4.1,-11.3,57.9,0.000,0.0,0.00,...,-0.017533,0.033904,-0.009691,0.011436,0.023780,0.005015,-0.008306,-0.001166,0.004274,0.008281
71447,2024-04-04 08:00:00,4,48.4753,35.0160,13.3,2.1,49.3,0.000,0.0,0.00,...,0.001341,-0.012739,-0.009141,0.018945,0.004047,-0.007199,-0.009027,-0.000717,0.003955,0.056174
35409,2023-03-02 13:00:00,3,50.7469,25.3263,1.0,-1.2,86.0,0.000,0.0,0.00,...,-0.038517,-0.023364,-0.014462,-0.042437,0.005619,0.019603,-0.019278,0.015453,0.006098,0.002165
277074,2023-07-24 02:00:00,13,49.8444,24.0254,22.7,15.2,64.1,0.100,100.0,4.17,...,-0.009246,0.012549,-0.046031,-0.002349,0.003011,-0.019112,0.043054,0.030388,-0.014154,-0.033854
207886,2024-09-21 22:00:00,9,48.9226,24.7147,13.8,8.2,72.8,0.000,0.0,0.00,...,-0.006482,-0.032010,-0.013145,-0.003005,-0.011898,-0.010269,0.002827,0.037000,0.011235,0.003838
40799,2023-10-13 04:00:00,3,50.7469,25.3263,14.4,11.9,85.7,6.078,100.0,4.17,...,-0.012126,0.023333,0.018984,-0.026396,-0.012972,-0.003734,-0.003493,-0.006728,0.004115,0.002877
510,2022-03-16 06:00:00,2,49.2336,28.4486,0.6,-6.2,62.1,0.000,0.0,0.00,...,-0.013425,0.015363,0.000603,0.002039,0.020398,-0.005536,-0.023336,-0.006609,0.014849,-0.017555
280147,2023-11-29 03:00:00,13,49.8444,24.0254,-4.0,-5.5,89.8,3.500,100.0,8.33,...,-0.016728,-0.024516,0.006396,0.032991,-0.024617,-0.003381,-0.017932,0.012893,0.014739,0.006208
429032,2022-10-11 09:00:00,19,49.5527,25.5889,9.4,5.2,76.1,0.000,0.0,0.00,...,-0.003718,-0.007037,-0.017598,0.004872,-0.022073,0.010494,-0.002956,0.019070,-0.036341,-0.029113
56364,2022-07-15 19:00:00,4,48.4753,35.0160,23.1,13.2,56.6,0.031,100.0,4.17,...,0.016152,-0.007920,0.003686,-0.035913,0.015931,-0.016474,-0.007263,-0.002716,0.000178,0.019265


In [16]:
df_final.isna().sum()

datetime_hour        0
region_id            0
city_latitude        0
city_longitude       0
day_temp             0
                  ... 
isw_topic_145     6336
isw_topic_146     6336
isw_topic_147     6336
isw_topic_148     6336
isw_topic_149     6336
Length: 216, dtype: int64

In [17]:
df_final.fillna(0, inplace=True)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635251,2025-03-01 19:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635252,2025-03-01 20:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635253,2025-03-01 21:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635254,2025-03-01 22:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664


In [18]:
df_final.isna().sum()

datetime_hour     0
region_id         0
city_latitude     0
city_longitude    0
day_temp          0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [19]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = (
    df_final.groupby('region_id')['isw_total_intensity']
    .diff(24)
    .fillna(0)
)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\1269306822.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\1269306822.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\1269306822.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [20]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
137235,2022-09-14 19:00:00,7,48.62636,22.28514,14.5,11.4,82.0,5.054,100.0,8.33,...,-0.012609,-0.016983,-0.035468,5.631793,0.065106,0.722903,0.037545,0.272076,5.565385,4.407911
568459,2023-08-02 12:00:00,24,48.29240,25.93550,21.0,17.3,80.5,6.631,100.0,8.33,...,-0.024590,-0.032298,-0.028953,5.257309,0.068243,0.778544,0.035049,-0.202450,5.277208,4.326901
634160,2025-01-15 08:00:00,26,50.45060,30.52430,0.6,-2.9,77.8,0.300,100.0,12.50,...,-0.014607,0.030087,0.001082,5.142954,0.067934,0.755589,0.034286,0.656233,4.809713,4.296594
46523,2024-06-07 17:00:00,3,50.74690,25.32630,19.0,15.0,78.9,1.402,100.0,8.33,...,-0.003214,0.003618,0.007885,4.906216,0.067623,0.756106,0.032708,0.352765,4.825793,4.277622
631623,2024-10-01 15:00:00,26,50.45060,30.52430,12.8,8.4,75.7,0.000,0.0,0.00,...,0.021695,0.026705,-0.021471,5.140233,0.068097,0.788891,0.034268,0.308776,5.030250,4.350591
51343,2024-12-25 13:00:00,3,50.74690,25.32630,-1.4,-2.5,92.3,0.000,0.0,0.00,...,-0.003441,0.006612,0.002684,4.302411,0.071704,0.776319,0.028683,-0.288311,4.380354,4.053878
184276,2025-01-19 01:00:00,8,47.82890,35.16260,2.6,-0.9,78.1,0.900,100.0,37.50,...,0.010932,0.000462,-0.006335,4.652272,0.072217,0.746543,0.031015,-0.057783,4.719923,4.013072
260107,2024-08-23 01:00:00,11,48.50850,32.26560,23.1,11.9,52.8,0.100,100.0,4.17,...,0.004391,0.007317,0.019726,4.481414,0.071064,0.790453,0.029876,0.281955,4.272424,4.116190
285768,2024-07-20 09:00:00,13,49.84440,24.02540,21.7,14.8,67.5,0.200,100.0,8.33,...,0.006606,-0.001812,0.011079,4.590563,0.070618,0.769547,0.030604,0.443445,4.430482,4.124909
279317,2023-10-25 13:00:00,13,49.84440,24.02540,13.1,11.8,91.6,18.000,100.0,8.33,...,-0.007774,-0.009385,-0.047869,4.897925,0.068940,0.776786,0.032653,0.563512,4.710194,4.254387


### TELEGRAM MERGE

In [21]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [22]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,...,0.017141,-0.013944,-0.027516,0.034612,-0.032061,-0.051141,-0.004707,-0.003985,-0.052194,-0.026334
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077087,-0.078417,0.091534,...,-0.019097,0.025730,0.000530,0.020332,-0.028041,0.001581,0.006948,-0.064965,0.022597,-0.017810
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053647,-0.076998,0.085930,...,0.025378,-0.003550,-0.043415,-0.007200,-0.009172,0.006124,-0.004550,0.020576,0.010544,0.012825
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059696,...,0.008728,-0.007502,-0.021433,0.028223,-0.025323,-0.045761,-0.001697,0.002329,-0.042282,-0.017344
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089562,-0.088811,0.109836,...,0.017978,0.027453,-0.019744,0.008659,0.023139,-0.009485,-0.030263,0.013689,-0.022727,0.006843


In [23]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [24]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]

tg_hourly = (
    df_tg_matrix.groupby("datetime_hour")[topic_cols]
    .mean()
    .sort_index()
    .ffill()
    .reset_index()
)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\2667129602.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


In [25]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,...,-0.013944,-0.027516,0.034612,-0.032061,-0.051141,-0.004707,-0.003985,-0.052194,-0.026334,2026-03-06 13:00:00
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077087,-0.078417,0.091534,...,0.025730,0.000530,0.020332,-0.028041,0.001581,0.006948,-0.064965,0.022597,-0.017810,2026-03-06 12:00:00
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053647,-0.076998,0.085930,...,-0.003550,-0.043415,-0.007200,-0.009172,0.006124,-0.004550,0.020576,0.010544,0.012825,2026-03-06 08:00:00
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059696,...,-0.007502,-0.021433,0.028223,-0.025323,-0.045761,-0.001697,0.002329,-0.042282,-0.017344,2026-03-05 16:00:00
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089562,-0.088811,0.109836,...,0.027453,-0.019744,0.008659,0.023139,-0.009485,-0.030263,0.013689,-0.022727,0.006843,2026-03-05 15:00:00


In [26]:
tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)

In [27]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [28]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
111125,2022-09-29 18:00:00,6,50.2536,28.6654,12.7,8.9,79.3,0.0,0.0,0.00,...,-0.000188,-0.018164,-0.006436,-0.012483,-0.012591,0.008418,-0.012232,0.006247,-0.002044,-0.010015
332111,2023-10-19 13:00:00,15,46.4725,30.7371,14.1,7.7,66.9,0.0,0.0,0.00,...,0.009710,-0.000538,-0.005392,0.001020,0.001894,-0.003435,-0.006466,0.006342,-0.002112,-0.009580
323208,2022-10-13 13:00:00,15,46.4725,30.7371,13.6,6.5,62.8,0.0,0.0,0.00,...,0.024201,-0.000644,-0.004013,-0.005897,-0.009958,-0.007205,-0.007659,0.011232,-0.000511,0.012045
29955,2022-07-18 07:00:00,3,50.7469,25.3263,16.6,9.9,67.0,0.0,0.0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129000,2024-10-13 15:00:00,6,50.2536,28.6654,8.6,4.7,77.3,0.0,0.0,0.00,...,0.019489,-0.028963,0.003416,0.009180,0.022108,-0.013390,0.010785,0.006369,0.015556,0.024905
487345,2023-05-23 09:00:00,21,46.6401,32.6142,18.8,13.4,72.9,0.2,100.0,4.17,...,0.002818,-0.010572,0.006213,0.010501,0.008953,0.013091,0.000803,-0.005289,-0.001634,-0.002477
548231,2024-04-18 14:00:00,23,49.4407,32.0637,9.0,8.3,94.9,9.0,100.0,75.00,...,-0.002060,-0.011072,0.005425,0.002511,-0.022067,-0.004982,0.010945,-0.003299,0.010573,0.001574
180918,2024-09-01 03:00:00,8,47.8289,35.1626,25.7,10.5,38.8,0.3,100.0,12.50,...,-0.013896,0.027705,-0.006205,0.014245,-0.012736,-0.004519,0.013263,0.018929,0.004659,0.024077
495334,2024-04-20 07:00:00,21,46.6401,32.6142,10.3,5.1,72.9,3.0,100.0,4.17,...,-0.026069,0.000635,0.005185,0.011609,0.002831,0.010996,0.001307,-0.014478,0.010474,0.017596
521139,2024-03-23 14:00:00,22,49.4168,26.9743,9.6,4.9,74.4,0.2,100.0,8.33,...,0.009563,0.003648,-0.002367,-0.017403,0.012339,-0.001015,-0.001502,0.002648,-0.007037,0.003978


In [29]:
df_final.shape

(635256, 473)

In [30]:
df_final = df_final.sort_values(["datetime_hour"])

In [31]:
with pd.option_context('display.max_rows', None):
    print(df_final.isnull().sum())

datetime_hour                         0
region_id                             0
city_latitude                         0
city_longitude                        0
day_temp                              0
day_dew                               0
day_humidity                          0
day_precip                            0
day_precipprob                        0
day_precipcover                       0
day_snow                              0
day_snowdepth                         0
day_windgust                          0
day_windspeed                         0
day_winddir                           0
day_pressure                          0
day_cloudcover                        0
day_solarradiation                    0
day_solarenergy                       0
day_uvindex                           0
day_moonphase                         0
hour_temp                             0
hour_feelslike                        0
hour_humidity                         0
hour_precip                           0


In [32]:
df_final = df_final.fillna(0)

In [33]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\409095542.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\409095542.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_26156\409095542.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [34]:
df_final.head(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,tg_total_intensity,tg_topic_std,tg_topic_max,tg_topic_entropy
0,2022-02-24,2,49.23360,28.44860,2.8,-0.3,80.5,0.300,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
582318,2022-02-24,25,51.49370,31.29440,2.4,-0.1,83.8,0.300,100.0,12.50,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
52938,2022-02-24,4,48.47530,35.01600,3.1,-2.2,70.6,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
555849,2022-02-24,24,48.29240,25.93550,3.4,-0.8,74.9,0.008,100.0,8.33,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
79407,2022-02-24,5,48.00200,37.81450,3.6,0.7,81.0,1.300,100.0,20.83,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
529380,2022-02-24,23,49.44070,32.06370,1.9,-0.6,83.4,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
105876,2022-02-24,6,50.25360,28.66540,3.3,0.2,80.8,0.800,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
502911,2022-02-24,22,49.41680,26.97430,2.3,-0.4,83.0,0.500,100.0,8.33,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
132345,2022-02-24,7,48.62636,22.28514,1.8,-1.6,79.9,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0
476442,2022-02-24,21,46.64010,32.61420,5.3,-1.0,65.8,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0


In [35]:
#df_final.head(50000).to_csv("df_final_50k.csv", index=False, encoding="utf-8-sig")